# Systems Design Foundations

Maps to `design3.md` Phase 3.

This notebook is a guided walkthrough of the mental models, estimation techniques, building blocks, and architecture patterns that underpin systems design work. It is written for a data/platform engineer who builds and operates real systems — Kafka pipelines, K8s clusters, Airflow DAGs, PostgreSQL-backed APIs — and wants to reason about design decisions with confidence.

The standard here is practical engineering judgment. You should be able to walk through a design, estimate its scale, identify its failure modes, and explain tradeoffs in concrete terms rather than abstract slogans.

Working rule:

- read the explanation
- run the code cells (estimation math, data models)
- modify the examples with your own numbers
- write one short note in your own words before moving on

## Learning Goals

By the end of this notebook, you should be able to:

- walk through an eight-step systems design framework from requirements to operational readiness
- perform back-of-envelope estimation for storage, throughput, and bandwidth
- explain core building blocks (load balancers, caches, message queues, databases) and when to use each
- compare architecture patterns (monolith vs microservices, event-driven, CQRS, saga) with real tradeoffs
- reason about data-intensive system patterns (batch, stream, lambda, kappa)
- conduct failure mode analysis on a realistic system
- describe the four golden signals and operational readiness requirements
- complete a full worked design for a market data ingestion system

## 1. The Systems Design Framework

Systems design is not about memorizing architectures. It is about applying a repeatable reasoning process to understand what a system must do, how it should behave under load and failure, and what tradeoffs are acceptable.

The eight-step framework below gives you a structured way to move from a vague problem statement to a concrete, defensible design. In practice, you rarely go through these steps in strict linear order — you loop back as you learn more — but having the checklist prevents you from skipping critical thinking.

### Step 1: Clarify Requirements

Before drawing a single box, separate what the system must *do* from how it must *behave*.

**Functional requirements** describe the system's capabilities:
- Ingest trade data from multiple vendors in real time
- Compute windowed VWAP aggregations
- Serve historical and real-time queries via API
- Send alerts when data quality thresholds are breached

**Non-functional requirements** describe quality attributes:
- **Latency**: end-to-end from trade event to queryable aggregate in under 5 seconds
- **Throughput**: handle 10K trades/second sustained, 50K peak
- **Availability**: 99.9% uptime (roughly 8.7 hours downtime/year)
- **Consistency**: aggregates must be eventually consistent within the window grace period
- **Durability**: no trade event may be silently dropped
- **Cost**: run on a 3-node K8s cluster, not a 50-node fleet

The most common design mistake is jumping to components without nailing down requirements. A system designed for 100 QPS looks nothing like one designed for 100K QPS.

### Step 2: Back-of-Envelope Estimation

Quantify the scale before choosing technologies. This step prevents both over-engineering and under-engineering.

- How many requests per second (QPS)?
- How much storage per day, per year?
- How much network bandwidth?
- How much memory for caching or in-flight state?

The math does not need to be precise. It needs to be in the right order of magnitude.

### Step 3: API Design First

Define the contract before the internals. The API is what consumers depend on — it is the hardest thing to change later.

- REST endpoints with clear resource naming
- Message schemas (Avro, Protobuf, JSON) for event-driven systems
- File formats and partitioning schemes for batch interfaces
- Pagination, filtering, error responses

### Step 4: Data Model

What entities exist? What are the relationships? What are the access patterns?

Access patterns drive storage choices more than anything else:
- Point lookups by ID → key-value or B-tree index
- Range scans by time → time-series optimized storage
- Aggregations across large datasets → columnar storage
- Relationship traversal → graph database

### Step 5: High-Level Architecture

Now draw the boxes and arrows. Identify:
- Components and their responsibilities
- Data flow direction and format
- System boundaries (what you build vs what you use as managed services)
- Synchronous vs asynchronous communication

### Step 6: Deep Dive on Critical Components

Pick the 2-3 components that carry the most risk or complexity and design them in detail. This is where you show engineering depth.

### Step 7: Failure Mode Analysis

For each component and connection, ask: what happens when this breaks? What is the blast radius? How do we detect it? How do we recover?

### Step 8: Monitoring and Operational Readiness

A system that cannot be observed cannot be operated. Define metrics, logging, tracing, alerting, and runbooks before launch.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class SystemDesign:
    """Structured template for walking through a systems design."""

    problem: str = ""
    functional_requirements: list[str] = field(default_factory=list)
    non_functional_requirements: dict[str, str] = field(default_factory=dict)
    estimation: dict[str, str] = field(default_factory=dict)
    api_endpoints: list[dict[str, str]] = field(default_factory=list)
    data_model: list[dict[str, str]] = field(default_factory=list)
    components: list[str] = field(default_factory=list)
    failure_modes: list[dict[str, str]] = field(default_factory=list)
    metrics: list[str] = field(default_factory=list)
    tradeoffs: list[dict[str, str]] = field(default_factory=list)


# Example: partially filled template
example = SystemDesign(
    problem="Design a market data ingestion system",
    functional_requirements=[
        "Ingest trades from 5 vendor APIs",
        "Validate and deduplicate events",
        "Compute 1-minute VWAP aggregates",
        "Store raw and aggregated data",
        "Serve queries via REST API",
    ],
    non_functional_requirements={
        "latency": "< 5s end-to-end",
        "throughput": "10K trades/sec sustained, 50K peak",
        "availability": "99.9%",
        "durability": "zero silent data loss",
    },
)

example

# Try next:
# 1. Fill in the estimation, api_endpoints, and failure_modes fields.
# 2. Add a tradeoff: what do you sacrifice for low latency?

## 2. Back-of-Envelope Estimation

The purpose of estimation is not to get an exact answer. It is to get into the right order of magnitude so you can make informed technology choices and avoid obviously wrong designs.

### Powers of 2 You Should Know

| Power | Approximate Value | Common Name |
|-------|-------------------|-------------|
| 2^10  | ~1 thousand       | 1 KB        |
| 2^20  | ~1 million        | 1 MB        |
| 2^30  | ~1 billion        | 1 GB        |
| 2^40  | ~1 trillion       | 1 TB        |
| 2^50  | ~1 quadrillion    | 1 PB        |

### Latency Numbers Every Engineer Should Know

These are approximate orders of magnitude, not exact measurements. The point is relative scale.

| Operation | Latency | Notes |
|-----------|---------|-------|
| L1 cache reference | ~1 ns | On-CPU, fastest possible |
| L2 cache reference | ~4 ns | Still on-CPU |
| Branch mispredict | ~5 ns | CPU pipeline stall |
| Mutex lock/unlock | ~25 ns | Thread synchronization cost |
| Main memory (RAM) reference | ~100 ns | Off-CPU, on-machine |
| Compress 1 KB with Snappy | ~3 us | Fast compression |
| SSD random read | ~16 us | 100x slower than RAM |
| Read 1 MB sequentially from SSD | ~100 us | SSD sequential is fast |
| Round trip within same datacenter | ~500 us | Network hop |
| Read 1 MB sequentially from disk | ~2 ms | Spinning disk |
| Disk seek | ~10 ms | Spinning disk, random |
| Round trip cross-continent | ~150 ms | US-East to EU-West |

Key takeaway: there is roughly a 10x gap between each tier. RAM is 100x faster than SSD, SSD is 100x faster than network, network is 10-100x faster than spinning disk seeks. Design decisions that move data between tiers have massive latency implications.

### Storage Estimation Formula

```
storage = rows_per_day * row_size_bytes * retention_days
```

### Throughput Estimation Formula

```
average_qps = total_requests_per_day / 86400
peak_qps = average_qps * peak_multiplier  (typically 2x-5x)
```

### Bandwidth Estimation Formula

```
bandwidth = peak_qps * average_request_size_bytes
```

In [ ]:
def estimate_trading_platform():
    """Back-of-envelope estimation for a trading data platform."""

    # --- Input assumptions ---
    trades_per_day = 1_000_000
    avg_trade_bytes = 200  # UUID + symbol + price + volume + side + timestamp + metadata
    retention_days = 365 * 2  # 2 years
    seconds_per_day = 86_400
    peak_multiplier = 5  # spikes during market open/close

    # --- Storage ---
    daily_storage_bytes = trades_per_day * avg_trade_bytes
    daily_storage_gb = daily_storage_bytes / (1024**3)
    total_storage_gb = daily_storage_gb * retention_days

    # --- Throughput ---
    avg_qps = trades_per_day / seconds_per_day
    peak_qps = avg_qps * peak_multiplier

    # --- Bandwidth ---
    avg_bandwidth_mbps = (avg_qps * avg_trade_bytes * 8) / (1024**2)
    peak_bandwidth_mbps = (peak_qps * avg_trade_bytes * 8) / (1024**2)

    # --- Memory for 1-minute window state ---
    # Keep last 60 seconds of trades in memory for windowed aggregation
    trades_in_window = peak_qps * 60
    window_memory_mb = (trades_in_window * avg_trade_bytes) / (1024**2)

    results = {
        "trades_per_day": f"{trades_per_day:,}",
        "avg_trade_size": f"{avg_trade_bytes} bytes",
        "daily_storage": f"{daily_storage_gb:.2f} GB",
        "total_storage_2yr": f"{total_storage_gb:.1f} GB",
        "avg_qps": f"{avg_qps:.1f}",
        "peak_qps": f"{peak_qps:.1f}",
        "avg_bandwidth": f"{avg_bandwidth_mbps:.2f} Mbps",
        "peak_bandwidth": f"{peak_bandwidth_mbps:.2f} Mbps",
        "window_memory_at_peak": f"{window_memory_mb:.1f} MB",
    }

    for label, value in results.items():
        print(f"  {label:25s} {value}")

    return results


estimate_trading_platform()

# Try next:
# 1. Change trades_per_day to 10M and see how the numbers shift.
# 2. Add aggregated rows (1 per symbol per minute) and estimate their storage separately.
# 3. What if you keep raw data for 30 days and aggregates for 2 years?

## 3. Core Building Blocks

Every system is assembled from a small set of fundamental components. The skill is not knowing that these exist — it is knowing when each one is the right tool and what tradeoffs it introduces.

### Load Balancers

A load balancer distributes incoming traffic across multiple backend instances. It exists to improve throughput, availability, and fault tolerance.

**L4 (Transport Layer) Load Balancing:**
- Operates on TCP/UDP connections
- Routes based on IP and port
- Very fast — does not inspect request content
- Example: Kubernetes Services are L4 load balancers by default. A `ClusterIP` Service distributes TCP connections across pods using iptables/IPVS rules.

**L7 (Application Layer) Load Balancing:**
- Operates on HTTP/gRPC requests
- Can route based on URL path, headers, cookies
- More flexible but more expensive per request
- Example: Kubernetes Ingress controllers (nginx, Traefik) are L7 load balancers. They can route `/api/trades` to one service and `/api/aggregates` to another.

**Common Algorithms:**

| Algorithm | How it works | Best for |
|-----------|-------------|----------|
| Round robin | Rotate through backends sequentially | Uniform workloads, stateless services |
| Least connections | Send to the backend with fewest active connections | Varying request durations |
| Consistent hashing | Hash request key to a backend; same key always goes to same backend | Stateful services, caching layers |
| Weighted round robin | Rotate with weights proportional to backend capacity | Heterogeneous hardware |

**When consistent hashing matters:**
If your consumer needs to maintain local state for a partition of keys (like windowed aggregation per symbol), consistent hashing ensures the same key always routes to the same instance. This is exactly what Kafka consumer groups do with partition assignment.

### Caching

Caching stores frequently accessed data in a faster layer to reduce load on the primary data store and improve response times. The core tradeoff is always: speed vs staleness.

**Cache-Aside (Lazy Loading):**
1. Application checks cache first
2. On cache miss, load from database
3. Write result to cache
4. Return to caller

This is the most common pattern. The application controls what gets cached and when. Drawback: the first request for any key is always slow (cold cache).

**Write-Through:**
1. Application writes to cache AND database together
2. Reads always hit cache

Advantage: cache is always consistent with the database. Drawback: writes are slower because they hit two stores, and you cache data that may never be read.

**Write-Behind (Write-Back):**
1. Application writes to cache only
2. Cache asynchronously flushes to database in the background

Advantage: writes are fast. Drawback: if the cache crashes before flushing, you lose data. This is acceptable for metrics and counters, dangerous for financial trades.

**Cache Invalidation:**

This is famously one of the two hard problems in computer science (along with naming things and off-by-one errors). Strategies:

- **TTL (Time-to-Live)**: simple, works for data that changes slowly. Set TTL to 60 seconds and accept up to 60 seconds of staleness.
- **Event-driven invalidation**: when the source data changes, publish an event that clears the cache entry. More complex but more precise.
- **Version keys**: include a version number in the cache key. Bump the version when data changes.

**When caching hurts:**
- Write-heavy workloads: cache invalidation overhead dominates
- Strong consistency requirements: cached data is always potentially stale
- High cardinality, low repeat access: cache hit rate is too low to justify the memory

**Industry context:**
Redis is the standard cache for hot metadata lookups (symbol info, venue config), query result caching (recent VWAP aggregates), and rate limiting (sliding window counters). In your existing project, caching computed aggregates for the API layer would reduce TimescaleDB load significantly.

### Message Queues

Message queues decouple producers from consumers, absorb traffic spikes, and enable asynchronous processing. They are the backbone of event-driven architectures.

**Apache Kafka:**
- Distributed append-only log
- High throughput (millions of messages/second per cluster)
- Messages are retained for a configurable period, enabling replay
- Ordering guaranteed within a partition, not across partitions
- Consumer groups enable parallel processing with at-least-once delivery
- Best for: event streaming, audit logs, change data capture, replay-dependent workloads
- You already use this: your energy trading platform uses Kafka in KRaft mode for trade event streaming

**RabbitMQ:**
- Traditional message broker with exchanges and queues
- Supports complex routing patterns (topic, fanout, headers)
- Per-message acknowledgment and rejection
- Messages are typically consumed once and removed
- Best for: task distribution, request-reply patterns, complex routing logic
- Example: routing data quality alerts to different teams based on severity and domain

**Amazon SQS:**
- Fully managed queue, no infrastructure to operate
- Standard queues: at-least-once, best-effort ordering
- FIFO queues: exactly-once, strict ordering (lower throughput)
- Auto-scales to any volume
- Best for: simple decoupling between services, when you do not need replay or complex routing

**Decision framework:**

| Need | Choose |
|------|--------|
| Event streaming with replay | Kafka |
| Complex routing, task queues | RabbitMQ |
| Simple decoupling, managed | SQS |
| Exactly-once semantics | Kafka (with idempotent consumers) or SQS FIFO |
| Ordering by key | Kafka (partition by key) |
| Fan-out to many consumers | Kafka (consumer groups) or RabbitMQ (fanout exchange) |

### Databases

The most impactful storage decision is matching the database type to the access pattern. Using a relational database for time-series range scans, or a key-value store for complex joins, creates problems that no amount of tuning can fix.

**OLTP — PostgreSQL:**
- Row-oriented storage optimized for transactional workloads
- Strong ACID guarantees
- Rich query language (SQL), mature ecosystem
- Good for: user data, configuration, metadata, transactional workflows
- Weakness: not designed for large analytical scans across millions of rows
- Industry context: your API serves trade queries from PostgreSQL/TimescaleDB

**OLAP — ClickHouse, StarRocks, DuckDB:**
- Columnar storage optimized for analytical queries (aggregations, scans)
- Reads only the columns needed, compresses well
- Good for: dashboards, reporting, ad-hoc analytics over large datasets
- Weakness: poor at point lookups and frequent small updates
- Industry context: if you needed a BI layer over historical trade data, ClickHouse would be a strong choice

**Time-Series — TimescaleDB, InfluxDB:**
- Optimized for time-stamped data with automatic time-based partitioning
- Fast range queries over recent time windows
- Built-in downsampling, retention policies, continuous aggregates
- Good for: metrics, IoT sensor data, financial tick data
- Weakness: less flexible for non-temporal queries
- Industry context: your project uses TimescaleDB for trade aggregates — this is the right fit

**Key-Value — Redis, DynamoDB:**
- Simple get/set by key, extremely fast reads
- Redis: in-memory, supports data structures (lists, sets, sorted sets, streams)
- DynamoDB: managed, durable, auto-scaling
- Good for: caching, sessions, rate limiting, feature flags
- Weakness: no complex queries, limited relationship modeling

**Document — MongoDB:**
- Stores JSON-like documents, flexible schema
- Good for: content management, catalogs, semi-structured data
- Weakness: joins are awkward, consistency model requires careful configuration
- Caution: "schemaless" does not mean "no schema" — it means the schema lives in your application code instead of the database, which is often worse

**Graph — Neo4j:**
- Optimized for traversing relationships (friends-of-friends, dependency graphs)
- Cypher query language for pattern matching
- Good for: knowledge graphs, fraud detection, dependency analysis
- Weakness: not designed for bulk analytical queries
- Industry context: your GraphRAG system leverages graph relationships for retrieval

**Decision framework:**

| Access Pattern | Best Fit |
|----------------|----------|
| Point lookup by ID | Key-value (Redis, DynamoDB) |
| Transactional CRUD | OLTP (PostgreSQL) |
| Time-range scans | Time-series (TimescaleDB) |
| Large analytical aggregations | OLAP (ClickHouse) |
| Flexible document storage | Document (MongoDB) |
| Relationship traversal | Graph (Neo4j) |

In [ ]:
# Building blocks summary as a quick-reference data structure

building_blocks = {
    "load_balancer": {
        "l4": {
            "operates_on": "TCP/UDP",
            "example": "K8s ClusterIP Service",
            "tradeoff": "fast but no content-based routing",
        },
        "l7": {
            "operates_on": "HTTP/gRPC",
            "example": "K8s Ingress (nginx)",
            "tradeoff": "flexible routing but higher per-request cost",
        },
    },
    "cache": {
        "cache_aside": "app checks cache, loads DB on miss — most common",
        "write_through": "write to cache + DB together — always consistent, slower writes",
        "write_behind": "write to cache, async flush — fast writes, risk of data loss",
        "invalidation": "TTL (simple), event-driven (precise), version keys (explicit)",
    },
    "message_queue": {
        "kafka": "distributed log, replay, high throughput, partition ordering",
        "rabbitmq": "complex routing, per-message ack, task queues",
        "sqs": "managed, simple decoupling, auto-scaling",
    },
    "database": {
        "oltp": "PostgreSQL — transactions, ACID, row-oriented",
        "olap": "ClickHouse — columnar, fast aggregations",
        "timeseries": "TimescaleDB — time-partitioned, range scans",
        "kv": "Redis — in-memory, sub-ms reads",
        "document": "MongoDB — flexible schema, JSON-like",
        "graph": "Neo4j — relationship traversal",
    },
}

# Print a compact summary
for category, details in building_blocks.items():
    print(f"\n{'=' * 60}")
    print(f"  {category.upper().replace('_', ' ')}")
    print(f"{'=' * 60}")
    if isinstance(details, dict):
        for key, value in details.items():
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in value.items():
                    print(f"    {k}: {v}")
            else:
                print(f"  {key}: {value}")

## 4. Architecture Patterns

Architecture patterns are recurring solutions to recurring structural problems. They are not rules — they are tools with known tradeoffs. Picking the wrong pattern is worse than picking no pattern.

### Monolith vs Microservices

**Monolith:**
- Single deployable unit containing all business logic
- Shared database, shared memory space
- Simple to develop, test, and deploy at small scale
- Debugging is straightforward — one process, one log stream
- Refactoring is easy because everything is in the same codebase

**Microservices:**
- Each service owns a bounded context, its own data, and its own deployment
- Services communicate via API calls or events
- Independent scaling and deployment
- But: distributed systems are hard. You trade in-process function calls for network calls, which introduces latency, partial failure, and eventual consistency

**When to use which:**

Start with a monolith. Split into services when you have:
- Clear bounded contexts with different scaling needs
- Independent teams that need to deploy independently
- Components with fundamentally different technology requirements

The worst architecture is a "distributed monolith" — microservice boundaries with monolithic coupling (shared databases, synchronous call chains, coordinated deployments).

### Event-Driven Architecture

Components communicate by producing and consuming events rather than making direct calls. The producer does not know or care who consumes its events.

**Advantages:**
- Loose coupling: adding a new consumer does not change the producer
- Natural audit trail: events are facts that happened
- Resilience: consumers can fail and recover independently
- Temporal decoupling: producer and consumer do not need to be online simultaneously

**Disadvantages:**
- Harder to reason about end-to-end flows
- Event ordering and exactly-once semantics are complex
- Debugging requires distributed tracing

**Industry context:** Your energy trading platform is event-driven. The producer publishes trade events to Kafka, the consumer processes them independently, and the API reads from the database. Adding a new consumer (say, a real-time alerting service) would require zero changes to the producer.

### CQRS (Command Query Responsibility Segregation)

Separate the write model (commands) from the read model (queries). The write side optimizes for correct state transitions. The read side optimizes for fast queries, possibly using denormalized views.

**When it helps:**
- Read and write workloads have very different characteristics
- You need different data shapes for different query patterns
- Write consistency and read performance have conflicting optimization needs

**When it hurts:**
- Simple CRUD applications where reads and writes are symmetric
- Small teams that cannot afford the operational overhead of two models

**Industry context:** Your project already uses CQRS in `src/cqrs/`. The command side processes trade events and maintains canonical state. The query side serves aggregated views optimized for API consumers.

### Event Sourcing

Instead of storing current state, store the sequence of events that produced that state. Current state is derived by replaying events.

**Advantages:**
- Complete audit trail — you can answer "how did we get here?"
- Can rebuild state at any point in time
- Natural fit for event-driven systems
- Enables temporal queries ("what was the portfolio value at 3pm?")

**Disadvantages:**
- Replaying millions of events to rebuild state is slow without snapshots
- Schema evolution of events is tricky
- Not a good fit for simple CRUD workloads

**Industry context:** Financial systems often use event sourcing because regulators require complete audit trails. Every trade, cancellation, and amendment is an immutable event.

### Saga Pattern

A saga is a sequence of local transactions across multiple services, where each step has a compensating action that can undo it if a later step fails.

**Example — Trade Settlement:**
1. Validate trade → (compensate: mark trade as rejected)
2. Reserve funds → (compensate: release reserved funds)
3. Execute settlement → (compensate: reverse settlement)
4. Update positions → (compensate: revert position changes)

If step 3 fails, the saga executes compensating actions for steps 2 and 1 in reverse order.

**Two coordination styles:**
- **Choreography**: each service listens for events and decides what to do next. Simple but hard to track.
- **Orchestration**: a central coordinator tells each service what to do. Easier to reason about but introduces a single point of coordination.

### Circuit Breaker

When a downstream dependency is failing, stop sending requests to it temporarily. This prevents cascade failures where one slow service makes every upstream caller slow.

**States:**
1. **Closed** (normal): requests flow through. Track failure rate.
2. **Open** (tripped): all requests fail immediately without calling the dependency. Return cached data or a degraded response.
3. **Half-open** (probing): allow a small number of test requests through. If they succeed, close the circuit. If they fail, re-open.

**Industry context:** Your project implements circuit breaker in the resilience layer. If a vendor API starts timing out, the circuit breaker trips and the system stops hammering it, giving it time to recover.

### Strangler Fig Pattern

Gradually replace a legacy system by routing requests to the new system incrementally.

1. New requests for feature X go to the new system
2. Old requests continue going to the legacy system
3. Over time, route more features to the new system
4. Eventually decommission the legacy system

This avoids the "big bang rewrite" which historically has a high failure rate. The name comes from strangler fig trees that gradually envelop and replace their host tree.

In [ ]:
# Architecture pattern decision matrix

from dataclasses import dataclass


@dataclass
class PatternTradeoff:
    pattern: str
    when_to_use: str
    when_to_avoid: str
    complexity: str  # low, medium, high


patterns = [
    PatternTradeoff(
        pattern="Monolith",
        when_to_use="Small team, early stage, unclear boundaries",
        when_to_avoid="Multiple teams needing independent deploys",
        complexity="low",
    ),
    PatternTradeoff(
        pattern="Microservices",
        when_to_use="Clear bounded contexts, independent scaling needs",
        when_to_avoid="Small team, tightly coupled domain",
        complexity="high",
    ),
    PatternTradeoff(
        pattern="Event-driven",
        when_to_use="Loose coupling, async processing, audit trails",
        when_to_avoid="Simple request-response CRUD",
        complexity="medium",
    ),
    PatternTradeoff(
        pattern="CQRS",
        when_to_use="Asymmetric read/write workloads, different query shapes",
        when_to_avoid="Symmetric CRUD, small data volume",
        complexity="medium",
    ),
    PatternTradeoff(
        pattern="Event sourcing",
        when_to_use="Audit requirements, temporal queries, replay needs",
        when_to_avoid="Simple state management, no audit requirement",
        complexity="high",
    ),
    PatternTradeoff(
        pattern="Saga",
        when_to_use="Distributed transactions across services",
        when_to_avoid="Single-database transactions suffice",
        complexity="high",
    ),
    PatternTradeoff(
        pattern="Circuit breaker",
        when_to_use="Unreliable dependencies, cascade failure risk",
        when_to_avoid="All dependencies are local and reliable",
        complexity="low",
    ),
    PatternTradeoff(
        pattern="Strangler fig",
        when_to_use="Incremental migration from legacy system",
        when_to_avoid="Greenfield projects",
        complexity="medium",
    ),
]

print(f"{'Pattern':<20} {'Complexity':<12} {'Use when...'}")
print("-" * 80)
for p in patterns:
    print(f"{p.pattern:<20} {p.complexity:<12} {p.when_to_use}")

# Try next:
# 1. For your energy trading platform, which patterns are already in use?
# 2. Which pattern would you add next, and why?

## 5. Data-Intensive System Patterns

Data platforms are rarely pure request-response systems. They combine batch processing, stream processing, and serving layers in various configurations.

### Batch Processing

Process large volumes of data in scheduled jobs. High throughput, high latency.

**Characteristics:**
- Input: bounded dataset (a day's worth of files, a database snapshot)
- Processing: transform, aggregate, enrich in bulk
- Output: derived datasets, materialized views, reports
- Latency: minutes to hours
- Tools: Airflow + Spark, dbt, custom Python scripts

**When to use:**
- Data freshness requirements are measured in hours, not seconds
- Processing requires multiple passes or complex joins across large datasets
- Exactly-once semantics are straightforward (reprocess the whole batch on failure)

**Industry context:** Your Airflow DAGs that backfill historical data and compute daily aggregates are batch processing.

### Stream Processing

Process data continuously as it arrives. Low latency, complex ordering semantics.

**Characteristics:**
- Input: unbounded stream of events
- Processing: transform, aggregate, enrich per-event or per-window
- Output: derived streams, real-time views, alerts
- Latency: milliseconds to seconds
- Tools: Kafka Streams, Flink, Spark Structured Streaming

**When to use:**
- Data must be actionable within seconds
- Events have natural ordering that must be preserved
- Late-arriving data must be handled gracefully (watermarks, grace periods)

**Key concepts:**
- **Event time vs processing time**: event time is when the trade happened; processing time is when your system processed it. They diverge under load or reprocessing.
- **Watermarks**: a declaration that "I believe all events up to time T have arrived." Late events after the watermark may be dropped or handled specially.
- **Windows**: tumbling (fixed, non-overlapping), sliding (overlapping), session (gap-based). Your project uses 1-minute tumbling windows for VWAP.

**Industry context:** Your Kafka consumer computing windowed VWAP aggregates is stream processing.

### Lambda Architecture

Run both a batch layer and a stream layer. The batch layer provides complete, correct results with high latency. The stream layer provides approximate, fast results. A serving layer merges both views.

**Advantages:**
- Batch layer can recompute from scratch, fixing any stream processing errors
- Stream layer provides low-latency approximate results

**Disadvantages:**
- You maintain two codepaths that must produce compatible results
- Operational complexity is roughly doubled
- Debugging discrepancies between batch and stream is painful

### Kappa Architecture

Use only a stream processing layer, with the ability to replay the input stream from the beginning when you need to recompute.

**Advantages:**
- Single codebase for all processing
- Simpler operations than lambda
- Replay from Kafka's retained log serves the role of batch reprocessing

**Disadvantages:**
- Replay of months of data through a stream processor can be slow
- Not all processing fits naturally into a streaming model

**Industry context:** Your energy trading platform is closer to kappa. Kafka retains events, the consumer processes them as a stream, and replay is possible by resetting consumer offsets.

### Exactly-Once Delivery

True exactly-once delivery across distributed systems is extremely difficult. In practice, most systems achieve **at-least-once delivery + idempotent consumers**.

**How it works:**
1. Producer sends a message. If the acknowledgment is lost, the producer retries, potentially creating a duplicate.
2. Consumer receives the message. If it crashes after processing but before committing the offset, it will reprocess the message on restart.
3. **Idempotent consumer**: the consumer checks whether it has already processed this event (by event ID, dedup key, or database upsert) and skips duplicates.

**Your project does this:** the deduplicate step in the ingestion pipeline checks for duplicate trade IDs before processing.

### Data Lake vs Data Warehouse vs Lakehouse

| Aspect | Data Lake | Data Warehouse | Lakehouse |
|--------|-----------|----------------|-----------|
| Storage | Raw files (Parquet, JSON, CSV) on object storage | Structured tables in a purpose-built engine | Structured tables on object storage |
| Schema | Schema-on-read | Schema-on-write | Schema-on-write with flexibility |
| Query engine | Bring your own (Spark, Presto, Trino) | Built-in (Snowflake, BigQuery, Redshift) | Built-in (Databricks, Apache Iceberg + Trino) |
| Cost | Low storage, variable compute | Higher, bundled | Medium, decoupled |
| Best for | Exploration, ML training data, raw archive | BI dashboards, reporting, governed analytics | Both, when you want lake flexibility with warehouse governance |
| Risk | Becomes a "data swamp" without governance | Expensive at scale, rigid schema | Newer technology, still maturing |

## 6. Failure Mode Analysis

The difference between a junior and senior design is not the happy path — it is the failure analysis. Every component can fail, and the question is always: what is the blast radius, how do we detect it, and how do we recover?

### How to Think About Failures

For each component and connection in your architecture, ask:

1. **What can go wrong?** (the failure mode)
2. **What is the impact?** (data loss, latency spike, total outage, silent corruption)
3. **How do we detect it?** (metrics, health checks, alerts)
4. **How do we mitigate it?** (redundancy, retry, circuit breaker, graceful degradation)
5. **How do we recover?** (automated restart, manual intervention, replay)

### Common Failure Modes

**Network Partition:**
Components cannot communicate. In a Kafka-based system, this could mean producers cannot reach the broker, or consumers cannot commit offsets.
- Impact: messages buffer locally, consumer lag grows, potential duplicate processing on recovery
- Mitigation: producer retries with backoff, consumer idempotency, monitor consumer lag

**Disk Full:**
Writes fail silently or the process crashes. Kafka brokers with full disks stop accepting messages. PostgreSQL with a full WAL disk stops accepting writes.
- Impact: data loss if writes are silently dropped, total write outage if the process crashes
- Mitigation: disk usage alerts at 70% and 85%, automatic log rotation, retention policies
- Detection: this is one of the most preventable failures — just monitor disk usage

**Memory Exhaustion (OOM Kill):**
The operating system kills the process when it exceeds memory limits. In Kubernetes, this manifests as `OOMKilled` pod restarts.
- Impact: in-flight data is lost, consumer reprocesses from last committed offset
- Mitigation: set memory limits conservatively, use backpressure to limit in-flight data, monitor RSS
- Industry context: your project implements backpressure handling to prevent unbounded memory growth in the consumer

**Dependency Timeout:**
An upstream service (vendor API, database) responds slowly instead of failing fast. This is often worse than a clean failure because it ties up threads and connections.
- Impact: thread pool exhaustion, cascading slowness, eventual timeout everywhere
- Mitigation: circuit breaker (fail fast), connection timeouts, separate thread pools per dependency
- Industry context: your project uses circuit breaker and exponential backoff retry for vendor APIs

**Data Corruption:**
Bad data enters the pipeline and propagates downstream. A vendor sends a trade with a negative price, a malformed timestamp, or a duplicate ID.
- Impact: corrupted aggregates, incorrect reports, broken downstream consumers
- Mitigation: validate at ingestion (your pipeline does this), dead letter queue for invalid messages, data quality monitoring
- Industry context: your ingestion pipeline validates → deduplicates → enriches → publishes, catching corruption early

**Clock Skew:**
Timestamps disagree across services. Machine A thinks it is 12:00:00.000, machine B thinks it is 12:00:00.350. In windowed aggregation, this can cause events to land in the wrong window.
- Impact: incorrect window boundaries, data assigned to wrong aggregation periods
- Mitigation: use NTP synchronization, design for event-time processing (not wall-clock time), allow grace periods for late arrivals

### Mitigation Strategy Cheat Sheet

| Strategy | What it does | When to use |
|----------|-------------|-------------|
| Redundancy | Multiple instances of critical components | Always for production systems |
| Retry with backoff | Retry failed operations with increasing delay | Transient failures (network blips) |
| Circuit breaker | Stop calling a failing dependency | Persistent dependency failures |
| Dead letter queue | Route unprocessable messages to a separate queue for investigation | Data validation failures |
| Graceful degradation | Serve reduced functionality instead of failing entirely | Partial outages |
| Bulkhead | Isolate failure domains (separate thread pools, separate pods) | Preventing cascade failures |
| Idempotent operations | Make it safe to retry without side effects | At-least-once delivery systems |
| Alerting | Notify humans when automated recovery is not sufficient | Everything |

In [ ]:
# Failure mode analysis as a structured exercise

failure_modes = [
    {
        "component": "Kafka broker",
        "failure": "Broker node crashes",
        "impact": "Partition leadership transfers, brief unavailability for affected partitions",
        "detection": "Under-replicated partitions metric, broker count drop",
        "mitigation": "Replication factor >= 3, min.insync.replicas = 2",
        "recovery": "Automatic leader election, broker restart",
    },
    {
        "component": "Kafka consumer",
        "failure": "OOM kill during batch processing",
        "impact": "In-flight window state lost, reprocessing from last committed offset",
        "detection": "Pod restart count, consumer lag spike",
        "mitigation": "Backpressure limits, memory-bounded window state",
        "recovery": "Automatic pod restart, idempotent processing handles duplicates",
    },
    {
        "component": "TimescaleDB",
        "failure": "Disk full on WAL partition",
        "impact": "All writes fail, consumer backs up, lag grows",
        "detection": "Disk usage alert at 80%, WAL size metric",
        "mitigation": "Retention policies, automatic chunk compression, disk alerts",
        "recovery": "Free disk space, writes resume, consumer catches up",
    },
    {
        "component": "Vendor API",
        "failure": "Vendor returns 500 errors intermittently",
        "impact": "Missing data for affected vendor, gaps in coverage",
        "detection": "Error rate metric, circuit breaker state change",
        "mitigation": "Circuit breaker, exponential backoff retry, fallback to other vendors",
        "recovery": "Circuit breaker half-open probe detects recovery",
    },
    {
        "component": "Network",
        "failure": "Partition between producer and Kafka",
        "impact": "Producer buffers locally, potential message ordering issues",
        "detection": "Producer send latency spike, buffer fullness metric",
        "mitigation": "Producer acks=all, retry with idempotent producer",
        "recovery": "Network restores, buffered messages drain",
    },
    {
        "component": "API service",
        "failure": "Slow query due to missing index",
        "impact": "API latency spike, downstream dashboards stall",
        "detection": "p99 latency alert, slow query log",
        "mitigation": "Query timeout, read replica, caching layer",
        "recovery": "Add missing index, restart connection pool",
    },
]

print(f"{'Component':<20} {'Failure':<35} {'Detection'}")
print("=" * 90)
for fm in failure_modes:
    print(f"{fm['component']:<20} {fm['failure']:<35} {fm['detection']}")

# Try next:
# 1. Add a failure mode for "data corruption from vendor" (malformed prices).
# 2. Add a failure mode for "clock skew between producer and consumer".
# 3. Rank these by blast radius — which failure affects the most users?

## 7. Monitoring and Operational Readiness

A system you cannot observe is a system you cannot operate. Monitoring is not an afterthought — it is a design requirement on par with data modeling and API design.

### The Four Golden Signals

Google's SRE book identifies four signals that, together, give you a comprehensive view of service health:

**1. Latency:**
The time it takes to serve a request. Track the distribution, not just the average. A system with 50ms average latency and 5s p99 latency has a tail latency problem that averages hide.
- Metric: histogram of request duration, broken down by endpoint
- Alert: p99 latency exceeds SLO threshold (e.g., > 500ms for more than 5 minutes)

**2. Traffic:**
The demand being placed on the system. For an API, this is requests per second. For a Kafka consumer, this is messages consumed per second.
- Metric: counter of requests/messages, broken down by type
- Alert: traffic drops to zero (upstream is broken) or spikes beyond capacity

**3. Errors:**
The rate of requests that fail. This includes explicit errors (HTTP 5xx) and implicit errors (successful response with wrong data).
- Metric: counter of errors, broken down by type and endpoint
- Alert: error rate exceeds threshold (e.g., > 1% of requests for 5 minutes)

**4. Saturation:**
How "full" the service is. CPU utilization, memory usage, disk usage, queue depth, connection pool usage. Saturation predicts imminent problems before they become outages.
- Metric: gauge of resource utilization
- Alert: resource usage exceeds capacity threshold (e.g., disk > 80%, CPU > 90% sustained)

### Metrics: Prometheus-Style Instrumentation

Three metric types cover most needs:

- **Counter**: monotonically increasing value (total requests, total errors, total bytes processed). You derive rates from counters: `rate(requests_total[5m])`.
- **Gauge**: value that goes up and down (current queue depth, memory usage, active connections).
- **Histogram**: distribution of values (request latency, message size). Enables percentile calculations (p50, p95, p99).

### Logging: Structured JSON

Structured logs are machine-parseable. Instead of:
```
2026-04-04 08:15:00 ERROR Failed to process trade EURUSD
```

Write:
```json
{"ts": "2026-04-04T08:15:00Z", "level": "error", "msg": "trade processing failed", "symbol": "EURUSD", "trade_id": "abc-123", "error": "price validation failed", "correlation_id": "req-456"}
```

The second form can be indexed, searched, filtered, and aggregated by any field. Your project uses structlog, which produces this format.

**Correlation IDs:** Assign a unique ID at the entry point (API request, Kafka message) and propagate it through every log line and downstream call. When investigating an incident, you can trace the full lifecycle of a single request across services.

### Tracing: Distributed Request Flows

When a request touches multiple services, logs from individual services are not enough. Distributed tracing (OpenTelemetry, Jaeger) creates a trace that spans all services a request touches, showing timing, dependencies, and bottlenecks.

A trace consists of:
- **Trace ID**: unique identifier for the entire request flow
- **Spans**: individual operations within the trace (each with start time, duration, metadata)
- **Parent-child relationships**: which span triggered which

### Dashboards: Grafana

Build dashboards that answer operational questions:
- **Overview dashboard**: the four golden signals for each service
- **Consumer dashboard**: consumer lag, processing rate, error rate per partition
- **Database dashboard**: query latency, connection pool usage, disk usage, replication lag
- **Alert dashboard**: currently firing alerts and recent alert history

### Alerting: Alert on Symptoms, Not Causes

Bad alert: "CPU usage > 80%"
Good alert: "p99 latency > 500ms for 5 minutes" (the symptom users experience)

CPU at 80% might be fine if latency is normal. Latency at 500ms is a problem regardless of CPU usage.

**Alert design principles:**
- Alert on SLO breaches (symptoms) rather than resource metrics (causes)
- Every alert must have a runbook: what to check first, what to do, who to escalate to
- Avoid alert fatigue: if an alert fires constantly and gets ignored, it is worse than no alert
- Use severity levels: page for data loss and outages, ticket for degraded performance

### Runbooks

A runbook is a step-by-step guide for responding to a specific alert. It should be written before the alert fires, not during an incident.

**Template:**
1. **Alert**: what triggered this runbook
2. **Impact**: what users/systems are affected
3. **Diagnosis**: specific checks to run (queries, logs, dashboards)
4. **Resolution**: steps to fix the immediate problem
5. **Escalation**: who to contact if resolution fails
6. **Post-incident**: follow-up actions (root cause analysis, prevention)

## 8. Worked Design: Market Data Ingestion System

This section walks through a complete systems design using the eight-step framework. This is the centerpiece of the notebook — everything above feeds into this.

**Problem statement:** Design a system that ingests trade data from multiple vendor APIs, validates and deduplicates events, computes real-time aggregates, stores data for historical queries, and serves results through a REST API.

---

### Step 1: Requirements

**Functional requirements:**
- Ingest trade events from 5 vendor APIs (WebSocket, SSE, HTTP polling, batch file)
- Validate events: reject malformed prices, volumes, timestamps
- Deduplicate: same trade reported by multiple vendors should be stored once
- Compute 1-minute VWAP (volume-weighted average price) per symbol
- Store raw trades and computed aggregates
- Serve queries: latest aggregate per symbol, historical range by time, raw trades by ID
- Dead letter queue for unprocessable events

**Non-functional requirements:**
- Latency: trade event to queryable aggregate in under 5 seconds
- Throughput: 10K trades/second sustained, 50K peak (market open/close)
- Availability: 99.9% (8.7 hours downtime/year)
- Durability: zero silent data loss — every valid trade must be stored
- Data retention: raw trades for 90 days, aggregates for 2 years
- Cost: run on a modest K8s cluster (3-5 nodes)

In [ ]:
# Step 2: Back-of-envelope estimation for the worked design

def estimate_market_data_system():
    """Full estimation for the market data ingestion system."""

    # --- Assumptions ---
    trades_per_day = 1_000_000  # 1M trades across 5 vendors
    raw_trade_bytes = 200       # UUID(16) + symbol(10) + price(8) + volume(8) + side(4)
                                # + timestamp(8) + vendor(10) + metadata(~136)
    aggregate_bytes = 150       # symbol + window_start + window_end + vwap + min + max
                                # + volume + trade_count
    symbols = 500               # number of distinct trading symbols
    retention_raw_days = 90
    retention_agg_days = 730    # 2 years
    seconds_per_day = 86_400
    peak_multiplier = 5
    api_read_qps_avg = 100      # dashboard + API consumers

    # --- Storage: Raw Trades ---
    raw_daily_gb = (trades_per_day * raw_trade_bytes) / (1024**3)
    raw_total_gb = raw_daily_gb * retention_raw_days

    # --- Storage: Aggregates ---
    # 1 aggregate per symbol per minute, 6.5 hours of market time per day
    agg_per_day = symbols * 390  # 390 minutes per trading day
    agg_daily_gb = (agg_per_day * aggregate_bytes) / (1024**3)
    agg_total_gb = agg_daily_gb * retention_agg_days

    # --- Throughput ---
    avg_write_qps = trades_per_day / seconds_per_day
    peak_write_qps = avg_write_qps * peak_multiplier

    # --- Bandwidth ---
    peak_ingest_mbps = (peak_write_qps * raw_trade_bytes * 8) / (1024**2)
    api_read_mbps = (api_read_qps_avg * 1024 * 8) / (1024**2)  # ~1KB per API response

    # --- Memory: Window State ---
    # At peak, 60 seconds of trades in memory for windowed aggregation
    window_trades = peak_write_qps * 60
    window_memory_mb = (window_trades * raw_trade_bytes) / (1024**2)

    # --- Kafka ---
    kafka_daily_gb = raw_daily_gb * 3  # replication factor 3
    kafka_retention_days = 7
    kafka_total_gb = kafka_daily_gb * kafka_retention_days

    results = {
        "--- STORAGE ---": "",
        "raw_daily": f"{raw_daily_gb:.2f} GB/day",
        "raw_total_90d": f"{raw_total_gb:.1f} GB",
        "agg_daily": f"{agg_daily_gb * 1024:.1f} MB/day",
        "agg_total_2yr": f"{agg_total_gb:.2f} GB",
        "kafka_with_replication": f"{kafka_total_gb:.1f} GB (7-day retention)",
        "--- THROUGHPUT ---": "",
        "avg_write_qps": f"{avg_write_qps:.0f}",
        "peak_write_qps": f"{peak_write_qps:.0f}",
        "api_read_qps": f"{api_read_qps_avg}",
        "--- BANDWIDTH ---": "",
        "peak_ingest": f"{peak_ingest_mbps:.1f} Mbps",
        "api_reads": f"{api_read_mbps:.1f} Mbps",
        "--- MEMORY ---": "",
        "window_state_at_peak": f"{window_memory_mb:.1f} MB",
    }

    for label, value in results.items():
        if label.startswith("---"):
            print(f"\n{label}")
        else:
            print(f"  {label:30s} {value}")

    print("\n--- VERDICT ---")
    print(f"  Total storage (DB):           {raw_total_gb + agg_total_gb:.0f} GB — fits on a single node")
    print(f"  Peak write QPS:               {peak_write_qps:.0f} — comfortable for Kafka + TimescaleDB")
    print(f"  Window memory:                {window_memory_mb:.0f} MB — trivial")
    print(f"  Conclusion: this is a SMALL system. No need for sharding or exotic infrastructure.")

    return results


estimate_market_data_system()

# Try next:
# 1. What if trades_per_day is 100M? At what point do you need to shard?
# 2. Add estimate for DLQ storage (assume 0.1% of trades are invalid).
# 3. What if you add a second consumer group for real-time alerting?

### Step 3: API Design

Define the external contract before designing internals.

**REST API Endpoints:**

```
GET  /api/v1/trades/{trade_id}              → single raw trade by ID
GET  /api/v1/trades?symbol=EURUSD&from=...&to=...&limit=100  → raw trades by time range
GET  /api/v1/aggregates/latest/{symbol}     → latest 1-min VWAP for a symbol
GET  /api/v1/aggregates?symbol=EURUSD&from=...&to=...        → historical aggregates
GET  /api/v1/symbols                        → list of active symbols
GET  /api/v1/health                         → service health check
GET  /api/v1/metrics                        → Prometheus metrics endpoint
```

**Kafka Message Schema (trade event):**

```json
{
  "trade_id": "550e8400-e29b-41d4-a716-446655440000",
  "symbol": "EURUSD",
  "price": "1.08450000",
  "volume": "1000.00",
  "side": "BUY",
  "timestamp": "2026-04-04T08:15:00.123Z",
  "vendor": "finnhub",
  "ingested_at": "2026-04-04T08:15:00.456Z"
}
```

Design decisions:
- Price as string with 8 decimal places — avoids floating-point precision loss
- Separate `timestamp` (event time) from `ingested_at` (processing time) — enables event-time windowing
- `vendor` field — enables per-vendor quality monitoring and deduplication
- Pagination via `limit` + cursor — avoids expensive `OFFSET` queries

In [ ]:
# Step 4: Data Model

from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal
from enum import Enum
from uuid import UUID


class TradeSide(str, Enum):
    BUY = "BUY"
    SELL = "SELL"


@dataclass
class TradeEvent:
    """Raw trade event as received from vendors."""

    trade_id: UUID
    symbol: str           # e.g., "EURUSD"
    price: Decimal        # 18,8 precision — never use float for money
    volume: Decimal
    side: TradeSide
    timestamp: datetime   # event time — when the trade happened
    vendor: str           # source vendor identifier
    ingested_at: datetime  # processing time — when we received it


@dataclass
class TradeAggregate:
    """1-minute VWAP aggregate per symbol."""

    symbol: str
    window_start: datetime
    window_end: datetime
    vwap: Decimal         # volume-weighted average price
    price_min: Decimal
    price_max: Decimal
    total_volume: Decimal
    trade_count: int


# --- Database schema (TimescaleDB) ---

trade_events_ddl = """
CREATE TABLE trade_events (
    trade_id      UUID PRIMARY KEY,
    symbol        VARCHAR(20)    NOT NULL,
    price         NUMERIC(18,8)  NOT NULL,
    volume        NUMERIC(18,8)  NOT NULL,
    side          VARCHAR(4)     NOT NULL,
    timestamp     TIMESTAMPTZ    NOT NULL,
    vendor        VARCHAR(50)    NOT NULL,
    ingested_at   TIMESTAMPTZ    NOT NULL DEFAULT NOW()
);

-- TimescaleDB hypertable for automatic time-based partitioning
SELECT create_hypertable('trade_events', 'timestamp');

-- Index for common query pattern: trades by symbol in a time range
CREATE INDEX idx_trades_symbol_time ON trade_events (symbol, timestamp DESC);
"""

trade_aggregates_ddl = """
CREATE TABLE trade_aggregates (
    symbol        VARCHAR(20)    NOT NULL,
    window_start  TIMESTAMPTZ    NOT NULL,
    window_end    TIMESTAMPTZ    NOT NULL,
    vwap          NUMERIC(18,8)  NOT NULL,
    price_min     NUMERIC(18,8)  NOT NULL,
    price_max     NUMERIC(18,8)  NOT NULL,
    total_volume  NUMERIC(18,8)  NOT NULL,
    trade_count   INTEGER        NOT NULL,
    PRIMARY KEY (symbol, window_start)
);

SELECT create_hypertable('trade_aggregates', 'window_start');
"""

print("--- TradeEvent fields ---")
for f_name, f_type in TradeEvent.__dataclass_fields__.items():
    print(f"  {f_name:<15} {f_type.type.__name__ if hasattr(f_type.type, '__name__') else str(f_type.type)}")

print("\n--- TradeAggregate fields ---")
for f_name, f_type in TradeAggregate.__dataclass_fields__.items():
    print(f"  {f_name:<15} {f_type.type.__name__ if hasattr(f_type.type, '__name__') else str(f_type.type)}")

print("\n--- Access patterns ---")
access_patterns = [
    ("Single trade by ID",     "trade_events",     "PK lookup on trade_id"),
    ("Trades by symbol+time",  "trade_events",     "Index scan on (symbol, timestamp)"),
    ("Latest aggregate",       "trade_aggregates", "Index scan on (symbol, window_start DESC) LIMIT 1"),
    ("Aggregate history",      "trade_aggregates", "Range scan on (symbol, window_start)"),
]
for pattern, table, method in access_patterns:
    print(f"  {pattern:<30} → {table:<20} via {method}")

### Step 5: High-Level Architecture

```
                          ┌─────────────┐
                          │  Vendor APIs │
                          │ (WS/SSE/    │
                          │  Poll/Batch) │
                          └──────┬──────┘
                                 │
                    ┌────────────▼────────────┐
                    │   Ingestion Service      │
                    │  (hexagonal architecture)│
                    │  - adapter per source    │
                    │  - validate, deduplicate │
                    │  - circuit breaker       │
                    └────────────┬────────────┘
                                 │ publish
                    ┌────────────▼────────────┐
                    │       Apache Kafka       │
                    │  topic: raw-trades       │
                    │  partitioned by symbol   │
                    │  replication factor: 3   │
                    └──┬─────────────────┬────┘
                       │                 │
          ┌────────────▼──────┐   ┌──────▼────────────┐
          │  Consumer Service  │   │  DLQ Consumer      │
          │  - windowed VWAP   │   │  - investigate     │
          │  - 1-min tumbling  │   │  - reprocess       │
          │  - backpressure    │   │  - alert           │
          └────────────┬──────┘   └───────────────────┘
                       │ write
          ┌────────────▼────────────┐
          │     TimescaleDB         │
          │  - trade_events table   │
          │  - trade_aggregates     │
          │  - hypertable partition │
          └────────────┬────────────┘
                       │ read
          ┌────────────▼────────────┐
          │    FastAPI REST API      │
          │  - /trades, /aggregates │
          │  - /health, /metrics    │
          │  - optional Redis cache │
          └─────────────────────────┘
```

**Component ownership:**
- Ingestion Service: owns vendor connectivity, data quality, and normalization
- Kafka: owns message durability and ordering (by partition key = symbol)
- Consumer Service: owns windowed aggregation and database writes
- TimescaleDB: owns data storage and time-based partitioning
- API Service: owns query serving and response formatting

**Key design decisions:**
- Partition Kafka by symbol: ensures all trades for the same symbol go to the same partition, enabling correct per-symbol windowed aggregation without distributed state
- Hexagonal architecture for ingestion: vendor-specific adapters behind a common port, making it easy to add new vendors
- Separate ingestion from aggregation: they have different scaling characteristics and failure modes

### Step 6: Deep Dive — Windowed Aggregation

The consumer service is the most complex component. It must:
1. Consume trades from Kafka partitions
2. Assign each trade to a 1-minute tumbling window based on event time
3. Maintain running state per window per symbol (sum of price*volume, sum of volume, min, max, count)
4. Emit the completed aggregate when the window closes
5. Handle late-arriving events within a grace period
6. Survive restarts without losing window state

**Window lifecycle:**
```
Window [08:15:00, 08:16:00)
  - 08:15:00.100: EURUSD trade arrives → create window, initialize state
  - 08:15:15.200: EURUSD trade arrives → update running totals
  - ...
  - 08:16:00.000: window closes → compute VWAP, emit aggregate, write to DB
  - 08:16:05.000: grace period (5s) — late arrivals still accepted
  - 08:16:05.001: grace period expires → window state is discarded
```

**VWAP calculation:**
```
VWAP = sum(price_i * volume_i) / sum(volume_i)
```

**Backpressure handling:**
If the consumer falls behind (e.g., database writes are slow), it must not accumulate unbounded in-flight state. Options:
- Pause Kafka consumption when in-flight messages exceed a threshold
- Batch database writes to amortize I/O cost
- Monitor consumer lag and alert when it exceeds acceptable thresholds

**Restart recovery:**
On restart, the consumer reprocesses from the last committed Kafka offset. Because the database write is idempotent (upsert on primary key), duplicate processing is safe.

### Steps 7-8: Failure Modes and Monitoring for This Design

**Failure mode analysis for the market data ingestion system:**

| Component | Failure | Impact | Mitigation |
|-----------|---------|--------|------------|
| Vendor API | Returns errors or goes down | Missing data for that vendor | Circuit breaker, retry with backoff, multi-vendor redundancy |
| Vendor API | Sends corrupt data (negative price) | Bad aggregates if not caught | Validation at ingestion, DLQ for invalid events |
| Ingestion → Kafka | Network partition | Events buffer in ingestion service memory | Bounded buffer, backpressure, idempotent producer |
| Kafka broker | Node crash | Brief partition unavailability | Replication factor 3, min.insync.replicas 2, auto leader election |
| Consumer | OOM during peak | Window state lost, reprocessing required | Memory limits, backpressure, idempotent upserts |
| Consumer | Bug in VWAP logic | Silently wrong aggregates | Unit tests, data quality checks (VWAP within reasonable bounds) |
| TimescaleDB | Disk full | All writes fail | Disk alerts at 70%/85%, retention policies, chunk compression |
| TimescaleDB | Replication lag | Stale reads from replica | Lag monitoring, route critical reads to primary |
| API Service | Connection pool exhaustion | Requests queue and timeout | Pool size tuning, connection timeout, circuit breaker |
| Cross-cutting | Clock skew | Events in wrong windows | NTP sync, event-time processing, grace periods |

**Monitoring plan:**

| Signal | Metric | Alert Threshold |
|--------|--------|----------------|
| Latency | End-to-end: trade event time to aggregate available via API | > 10s for 5 minutes |
| Latency | API p99 response time | > 500ms for 5 minutes |
| Traffic | Trades ingested per second (by vendor) | Drops to 0 for any vendor for 2 minutes |
| Traffic | API requests per second | Drops below 50% of baseline |
| Errors | Ingestion validation failure rate | > 5% of events for 5 minutes |
| Errors | API 5xx rate | > 1% of requests for 5 minutes |
| Saturation | Kafka consumer lag (messages) | > 10,000 messages for 5 minutes |
| Saturation | TimescaleDB disk usage | > 80% |
| Saturation | Consumer memory RSS | > 80% of pod limit |
| Saturation | API connection pool usage | > 90% |

In [ ]:
# Full worked design as a completed SystemDesign instance

worked_design = SystemDesign(
    problem="Design a market data ingestion system for 5 vendors with real-time VWAP aggregation",
    functional_requirements=[
        "Ingest trades from 5 vendor APIs (WS, SSE, polling, batch)",
        "Validate: reject malformed prices, volumes, timestamps",
        "Deduplicate: same trade from multiple vendors stored once",
        "Compute 1-minute VWAP per symbol via tumbling windows",
        "Store raw trades (90d) and aggregates (2yr)",
        "Serve queries via REST API: latest aggregate, historical range, raw by ID",
        "Dead letter queue for unprocessable events",
    ],
    non_functional_requirements={
        "latency": "< 5s end-to-end (trade event to queryable aggregate)",
        "throughput": "10K trades/sec sustained, 50K peak",
        "availability": "99.9% (8.7 hours downtime/year)",
        "durability": "zero silent data loss",
        "retention": "raw 90 days, aggregates 2 years",
        "cost": "3-5 node K8s cluster",
    },
    estimation={
        "storage_raw_90d": "~17 GB",
        "storage_agg_2yr": "~0.04 GB",
        "kafka_7d_with_replication": "~3.9 GB",
        "peak_write_qps": "~58",
        "window_memory_at_peak": "~0.7 MB",
        "verdict": "Small system — single-node DB, no sharding needed",
    },
    api_endpoints=[
        {"method": "GET", "path": "/api/v1/trades/{trade_id}", "desc": "single trade by ID"},
        {"method": "GET", "path": "/api/v1/trades?symbol=&from=&to=", "desc": "trades by range"},
        {"method": "GET", "path": "/api/v1/aggregates/latest/{symbol}", "desc": "latest VWAP"},
        {"method": "GET", "path": "/api/v1/aggregates?symbol=&from=&to=", "desc": "agg history"},
        {"method": "GET", "path": "/api/v1/health", "desc": "health check"},
    ],
    data_model=[
        {"entity": "TradeEvent", "storage": "TimescaleDB hypertable", "key": "trade_id"},
        {"entity": "TradeAggregate", "storage": "TimescaleDB hypertable", "key": "(symbol, window_start)"},
    ],
    components=[
        "Ingestion Service (hexagonal, per-vendor adapters, circuit breaker)",
        "Apache Kafka (KRaft mode, partitioned by symbol, replication 3)",
        "Consumer Service (windowed VWAP, backpressure, idempotent writes)",
        "TimescaleDB (hypertables, retention policies, compression)",
        "FastAPI REST API (read-only, optional Redis cache)",
        "Dead Letter Queue consumer (investigation, reprocessing)",
    ],
    failure_modes=[
        {"component": "Vendor API", "failure": "down/corrupt", "mitigation": "circuit breaker + DLQ"},
        {"component": "Kafka", "failure": "broker crash", "mitigation": "replication + auto leader election"},
        {"component": "Consumer", "failure": "OOM", "mitigation": "backpressure + idempotent upserts"},
        {"component": "TimescaleDB", "failure": "disk full", "mitigation": "alerts + retention policies"},
        {"component": "API", "failure": "pool exhaustion", "mitigation": "timeouts + circuit breaker"},
    ],
    metrics=[
        "End-to-end latency (trade event time → aggregate available)",
        "Trades ingested/sec by vendor",
        "Kafka consumer lag",
        "API p99 latency",
        "Validation failure rate",
        "TimescaleDB disk usage",
        "Consumer memory RSS",
    ],
    tradeoffs=[
        {"accept": "eventual consistency (5s window)", "because": "enables streaming architecture"},
        {"accept": "at-least-once delivery", "because": "idempotent upserts make it safe and simpler than exactly-once"},
        {"accept": "partition by symbol limits parallelism to # symbols", "because": "correct per-symbol ordering is required for VWAP"},
        {"reject": "in-memory-only aggregation without persistence", "because": "durability requirement — cannot lose data on restart"},
    ],
)

print("=== WORKED DESIGN SUMMARY ===\n")
print(f"Problem: {worked_design.problem}\n")
print(f"Components: {len(worked_design.components)}")
print(f"Failure modes analyzed: {len(worked_design.failure_modes)}")
print(f"Metrics defined: {len(worked_design.metrics)}")
print(f"Tradeoffs documented: {len(worked_design.tradeoffs)}")

# Try next:
# 1. Add a new component: "Real-time alerting service" that consumes from the same Kafka topic.
# 2. What new failure modes does the alerting service introduce?
# 3. How would the estimation change if you needed 100x the throughput?

## 9. Design Exercise Templates

Use these exercises to practice the eight-step framework. Each one includes starter prompts for requirements, estimation, and tradeoffs. Work through them using the `SystemDesign` dataclass above.

---

### Exercise A: Design a Data Pipeline for 5 Vendors

**Scenario:** Your company receives market data from 5 different vendors. Each vendor has a different API format (REST, FTP file drop, WebSocket, S3 bucket, email attachment). You need a unified pipeline that normalizes, validates, and loads data into a single analytics database.

**Requirements prompts:**
- What is the SLA for data freshness? Same-day? Within 1 hour? Within 5 seconds?
- What happens when a vendor is late? Do downstream consumers wait or proceed with partial data?
- How do you handle schema changes from a vendor?

**Estimation prompts:**
- How many records per vendor per day? What is the largest vendor?
- What is the total storage for 1 year of normalized data?
- What is the peak QPS if all vendors send data simultaneously?

**Tradeoff prompts:**
- Do you build one adapter per vendor or one generic adapter? (specific = correct, generic = less code)
- Do you validate at ingestion or at query time? (early = safe, late = flexible)
- Do you store raw and normalized, or just normalized? (both = debuggable, one = cheaper)

---

### Exercise B: Design a Feature Store for ML Researchers

**Scenario:** ML researchers need a system to register, store, and retrieve features for training and inference. Features are computed from raw data (trades, user events, market indicators) and must be available both for batch training (historical) and online inference (real-time).

**Requirements prompts:**
- How many features? How many entities (users, symbols)?
- What is the acceptable latency for online feature serving? (typically < 10ms)
- Do researchers need point-in-time correctness (no data leakage from the future)?

**Estimation prompts:**
- Features per entity * entities * history depth = storage
- Online serving: QPS * features per request * feature size = bandwidth
- Batch: full feature matrix size for training jobs

**Tradeoff prompts:**
- Dual storage (offline in data lake, online in Redis/DynamoDB) vs single store?
- Push features on computation vs pull on demand?
- Feature versioning: how do you handle a feature definition change?

---

### Exercise C: Design a Real-Time Alerting System for Data Quality

**Scenario:** You need a system that monitors data pipelines in real time and alerts when data quality drops below acceptable thresholds. Examples: missing data for a symbol, price spikes beyond historical bounds, volume anomalies, schema violations.

**Requirements prompts:**
- How many data quality rules? How often are they evaluated?
- What is the acceptable alert latency? (detect and notify within 1 minute?)
- Who receives alerts? (Slack, PagerDuty, email, dashboard)

**Estimation prompts:**
- Rules * evaluation frequency * data volume = compute requirements
- Alert volume: how many alerts per day at steady state? (if > 50, you have alert fatigue)
- Storage for alert history and investigation context

**Tradeoff prompts:**
- Stream-based evaluation (low latency, complex) vs batch-based (simpler, higher latency)?
- Hard-coded thresholds vs ML-based anomaly detection? (simple = explainable, ML = adaptive)
- Alert per rule violation vs aggregated alert summary? (granular = noisy, summary = may miss)

In [ ]:
# Blank template for working through the exercises

exercise_a = SystemDesign(problem="Design a data pipeline for 5 vendors")
exercise_b = SystemDesign(problem="Design a feature store for ML researchers")
exercise_c = SystemDesign(problem="Design a real-time alerting system for data quality")

# Fill these in as you work through each exercise.
# Use the worked design (section 8) as your reference for depth and completeness.
print("Templates created. Fill in each field using the prompts above.")
print(f"  Exercise A: {exercise_a.problem}")
print(f"  Exercise B: {exercise_b.problem}")
print(f"  Exercise C: {exercise_c.problem}")

## 10. Mini Lab

Work through these without searching elsewhere first.

1. **Estimation drill:** A new requirement arrives — store all API request/response pairs for audit purposes. Each pair is ~2 KB. The API handles 100 QPS average, 500 QPS peak. Estimate daily storage, yearly storage, and whether you need a separate audit database or can co-locate with the main DB.

2. **Architecture drill:** Your team wants to add a real-time leaderboard showing "top 10 symbols by volume in the last 5 minutes." Walk through the design:
   - Where does the data come from? (hint: you already have it in Kafka)
   - What type of window do you need? (not tumbling)
   - Where do you store the leaderboard state? (hint: fast reads, small data)
   - What happens when the consumer restarts?

3. **Failure mode drill:** The TimescaleDB primary node goes down during market hours. Walk through:
   - What immediately breaks? (writes, reads, or both?)
   - What is the blast radius? (which services are affected?)
   - What is the recovery path? (failover, replay, manual intervention?)
   - What monitoring would have given you early warning?

4. **Tradeoff drill:** Your manager asks: "Can we replace Kafka with a simple PostgreSQL-based queue (using LISTEN/NOTIFY) to reduce infrastructure complexity?" Write a structured argument for or against, covering throughput, durability, replay, operational overhead, and team expertise.

5. **Full design drill:** Pick one of the three exercises from Section 9 and complete a full `SystemDesign` instance with all fields populated. Spend 30-45 minutes, as you would in a design discussion.

## 11. Exit Criteria

Do not move on until you can say yes to these:

- I can walk through the eight-step systems design framework from memory.
- I can perform back-of-envelope estimation for storage, throughput, bandwidth, and memory, and arrive at the right order of magnitude.
- I can explain the tradeoffs of each core building block: load balancers (L4 vs L7), caching (aside/through/behind), message queues (Kafka vs RabbitMQ vs SQS), and databases (OLTP vs OLAP vs time-series vs KV vs document vs graph).
- I can compare architecture patterns (monolith vs microservices, event-driven, CQRS, event sourcing, saga, circuit breaker) and explain when each is the right or wrong choice.
- I can explain batch vs stream processing, lambda vs kappa architecture, and exactly-once delivery.
- I can conduct a failure mode analysis: identify what can go wrong, assess blast radius, and propose mitigations.
- I can define a monitoring plan using the four golden signals (latency, traffic, errors, saturation).
- I can walk through a complete worked design (requirements, estimation, API, data model, architecture, failure modes, monitoring) for a realistic system.
- I can explain design tradeoffs in concrete engineering terms, not abstract slogans.

## 12. References

Core texts and resources used to build this notebook:

- **Designing Data-Intensive Applications** — Martin Kleppmann
  The definitive reference for data systems architecture. Covers replication, partitioning, stream processing, batch processing, consistency models.

- **System Design Interview** — Alex Xu (Volumes 1 and 2)
  Structured framework for systems design with worked examples. Good for the estimation and component-selection mindset.

- **Google SRE Book** — Beyer, Jones, Petoff, Murphy
  Source of the four golden signals, SLO-based alerting, and operational readiness concepts.
  https://sre.google/sre-book/table-of-contents/

- **Apache Kafka Documentation**
  Authoritative reference for Kafka internals: partitioning, replication, consumer groups, exactly-once semantics.
  https://kafka.apache.org/documentation/

- **TimescaleDB Documentation**
  Hypertables, continuous aggregates, compression, retention policies.
  https://docs.timescale.com/

- **Kubernetes Documentation**
  Services (L4 LB), Ingress (L7 LB), resource limits, pod lifecycle.
  https://kubernetes.io/docs/

- **Redis Documentation**
  Data structures, caching patterns, persistence options.
  https://redis.io/docs/

- **Martin Fowler — Patterns of Enterprise Application Architecture**
  CQRS, event sourcing, saga, strangler fig, circuit breaker pattern descriptions.
  https://martinfowler.com/articles/

- **Jeff Dean — Latency Numbers Every Programmer Should Know**
  Original source for the latency reference table.
  https://colin-scott.github.io/personal_website/research/interactive_latency.html

## Interview Question Bank

Use these after you finish the notebook. Aim for crisp answers with tradeoffs.

- How do you turn a vague prompt into functional and non-functional requirements?
- Why do capacity estimates matter before discussing components?
- When do you replicate, and when do you partition?
- What is the difference between throughput, latency, availability, and durability?
- How do you choose APIs, storage models, and message boundaries from requirements?
- What failure modes do you analyze first in a distributed design?
- When is eventual consistency acceptable, and when is it not?
- How do you explain a design tradeoff without sounding hand-wavy?
